In [1]:
import os
import logging
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:8]}")
else:
    print("OpenRouter API Key not set")

OpenRouter API Key exists and begins sk-or-v1


In [3]:
openrouter_url = "https://openrouter.ai/api/v1"
openrouter = OpenAI(api_key=openrouter_api_key,base_url=openrouter_url)
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [4]:
xiaomi_flash_model = "xiaomi/mimo-v2-flash"
deepseek_chat_model = "deepseek/deepseek-chat"
deepseek_model_3_2 = "deepseek/deepseek-v3.2"
mistralai_devstral_2512 = "mistralai/devstral-2512"
mistralai_mixtral_8x7b_instruct = "mistralai/mixtral-8x7b-instruct"
gemma4 = "google/gemma-4-31b-it"
qwen_qwen3_6_plus_free = "qwen/qwen3.5-397b-a17b"



In [5]:
DB_FILE = "school.db"
#Global connection variable
conn = None

In [6]:
system_message = """You are an AI assistant designed to help teachers analyze student performance for parent-teacher meetings.

Your responsibilities:
- Understand the teacher’s question accurately
- Determine what type of academic data is needed
- Use the correct available tool
- Analyze returned data carefully
- Provide concise, teacher-friendly insights

Available tools:
1. get_marks(student_id)
   - Use for general marks requests when no specific timeframe is mentioned

2. get_marks_history(student_id)
   - Use for trends, improvement, decline, historical performance, or comparison over time

3. get_latest_marks(student_id)
   - Use for current performance, latest exam, recent scores, or present weaknesses

4. get_attendance(student_id)
   - Use when attendance or consistency may be relevant

Rules for tool usage:
- If student data is required, you MUST call the correct tool before answering
- Never guess or assume marks, attendance, or trends
- You may call multiple tools when needed
- Always wait for tool results before final analysis
- If the user asks about:
  • "improvement", "trend", "history", "decline" → use get_marks_history
  • "latest", "current", "recent", "now" → use get_latest_marks
  • general marks → use get_marks

Analysis requirements:
After receiving tool results:
- Identify strengths (best-performing subjects)
- Identify weaknesses (lowest-performing subjects)
- Mention trend if historical data is available
- Mention attendance impact if attendance is low
- Be objective and factual

Response style:
- Keep responses short (4–6 lines)
- Use simple language suitable for teachers and parents
- Be specific, not generic
- Avoid exaggeration
- If data is incomplete, clearly mention missing information

For summary requests, ALWAYS include:
1. Strength
2. Weakness
3. Attendance
4. Improvement suggestion

Example:
“Rahul performs strongly in Maths but needs improvement in English grammar. His recent scores show steady Maths performance but weaker language consistency. Attendance is 75%, which may be affecting overall progress. Daily reading and grammar practice are recommended.”

Your goal:
Help teachers quickly understand student performance, identify academic needs, and communicate effectively with parents."""

In [7]:
import sqlite3
from sqlite3 import Error


def create_connection(db_file):
    """Create a database connection"""
    try:
        conn = sqlite3.connect(db_file)
        conn.execute("PRAGMA foreign_keys = ON;")  # Enforce foreign keys
        print(f"Database is created.")
        return conn
    except Error as e:
        print(f"Database connection error: {e}")
        return None


def create_tables(conn):
    """Create tables if they do not exist"""
    try:
        cursor = conn.cursor()
        print("Create table enter ...")

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS students (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL
        );
        """)

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS subjects (
            id INTEGER PRIMARY KEY,
            name TEXT UNIQUE NOT NULL
        );
        """)

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS exam_types (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            max_marks INTEGER NOT NULL
        );
        """)

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS exams (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            exam_type_id INTEGER NOT NULL,
            date TEXT,
            FOREIGN KEY (exam_type_id) REFERENCES exam_types(id)
        );
        """)

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS marks (
            id INTEGER PRIMARY KEY,
            student_id INTEGER NOT NULL,
            subject_id INTEGER NOT NULL,
            exam_id INTEGER NOT NULL,
            marks_obtained REAL NOT NULL,
            FOREIGN KEY (student_id) REFERENCES students(id),
            FOREIGN KEY (subject_id) REFERENCES subjects(id),
            FOREIGN KEY (exam_id) REFERENCES exams(id)
        );
        """)

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS mark_components (
            id INTEGER PRIMARY KEY,
            mark_id INTEGER NOT NULL,
            component_name TEXT NOT NULL,
            max_marks REAL NOT NULL,
            marks_obtained REAL NOT NULL,
            FOREIGN KEY (mark_id) REFERENCES marks(id)
        );
        """)

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS attendance (
            id INTEGER PRIMARY KEY,
            student_id INTEGER NOT NULL,
            month TEXT,
            working_days INTEGER,
            present_days INTEGER,
            percentage REAL,
            FOREIGN KEY (student_id) REFERENCES students(id)
        );
        """)

        # Prevent duplicates when mock data is inserted multiple times.
        cursor.execute("""
        CREATE UNIQUE INDEX IF NOT EXISTS idx_marks_unique
        ON marks(student_id, subject_id, exam_id);
        """)

        conn.commit()
        print("Tables created successfully (if not already existing).")

    except Error as e:
        print(f"Error while creating tables: {e}")
        conn.rollback()


In [8]:
#insert mock data dont run this below insert mock data
# import sqlite3
# from sqlite3 import Error


# def insert_mock_data(conn):
#     """Insert mock data safely"""
#     try:
#         cursor = conn.cursor()

#         # STUDENTS
#         students = [
#             (101, 'Rahul'),
#             (102, 'Anita'),
#             (103, 'Kumar')
#         ]
#         cursor.executemany("""
#             INSERT OR IGNORE INTO students (id, name)
#             VALUES (?, ?)
#         """, students)

#         # SUBJECTS
#         subjects = [
#             (1, 'Maths'),
#             (2, 'Tamil'),
#             (3, 'Science'),
#             (4, 'Social'),
#             (5, 'English'),
#             (6, 'AI')
#         ]
#         cursor.executemany("""
#             INSERT OR IGNORE INTO subjects (id, name)
#             VALUES (?, ?)
#         """, subjects)

#         # EXAM TYPES
#         exam_types = [
#             (1, 'Unit Test', 20),
#             (2, 'Midterm', 40),
#             (3, 'Final Exam', 100)
#         ]
#         cursor.executemany("""
#             INSERT OR IGNORE INTO exam_types (id, name, max_marks)
#             VALUES (?, ?, ?)
#         """, exam_types)

#         # EXAMS
#         exams = [
#             (1, 'Unit Test 1', 1, '2024-01-10'),
#             (2, 'Unit Test 2', 1, '2024-02-10'),
#             (3, 'Unit Test 3', 1, '2024-03-10'),
#             (4, 'Pre Midterm', 2, '2024-06-15'),
#             (5, 'Post Midterm', 2, '2024-07-15'),
#             (6, 'Half Yearly', 3, '2024-09-10'),
#             (7, 'Annual Exam', 3, '2025-02-10')
#         ]
#         cursor.executemany("""
#             INSERT OR IGNORE INTO exams (id, name, exam_type_id, date)
#             VALUES (?, ?, ?, ?)
#         """, exams)

#         # MARKS
#         marks = [
#             # Rahul
#             (101, 1, 1, 18), (101, 1, 2, 17), (101, 1, 3, 19), (101, 1, 6, 78),
#             (101, 5, 1, 10), (101, 5, 2, 12), (101, 5, 3, 11), (101, 5, 6, 55),
#             (101, 3, 1, 15), (101, 3, 2, 16), (101, 3, 3, 17), (101, 3, 6, 70),

#             # Anita
#             (102, 1, 1, 12), (102, 1, 2, 14), (102, 1, 3, 15), (102, 1, 6, 65),
#             (102, 5, 1, 18), (102, 5, 2, 19), (102, 5, 3, 17), (102, 5, 6, 80),

#             # Kumar
#             (103, 1, 1, 8), (103, 1, 2, 10), (103, 1, 3, 9), (103, 1, 6, 50),
#             (103, 5, 1, 6), (103, 5, 2, 7), (103, 5, 3, 8), (103, 5, 6, 45)
#         ]
#         cursor.executemany("""
#             INSERT OR IGNORE INTO marks (student_id, subject_id, exam_id, marks_obtained)
#             VALUES (?, ?, ?, ?)
#         """, marks)

#         # mark_components
#         mark_components = [
#             (4, 'written_exam', 80, 62),
#             (4, 'class_test', 5, 5),
#             (4, 'activity', 5, 4),
#             (4, 'assignment', 5, 4),
#             (4, 'project', 5, 3)
#         ]

#         cursor.executemany("""
#             INSERT OR IGNORE INTO mark_components (mark_id, component_name, max_marks, marks_obtained)
#             VALUES (?, ?, ?, ?)
#         """, mark_components)


#         # attendance
#         attendance = [
#             (101, '2024-01', 22, 18, 81.8),
#             (101, '2024-02', 20, 15, 75.0),
#             (102, '2024-01', 22, 21, 95.4),
#             (103, '2024-01', 22, 16, 72.7)
#         ]

#         cursor.executemany("""
#             INSERT OR IGNORE INTO attendance (student_id, month, working_days, present_days, percentage)
#             VALUES (?, ?, ?, ?, ?)
#         """, attendance)

#         conn.commit()
#         print("Mock data inserted successfully.")

#     except Error as e:
#         print(f"Error inserting mock data: {e}")
#         conn.rollback()

#     except Exception as e:
#         print(f"Unexpected error: {e}")
#         conn.rollback()

In [9]:
import random
from sqlite3 import Error


def insert_mock_data(conn):
    """Insert production-ready school data for 40 students"""
    try:
        cursor = conn.cursor()

        # -----------------------------
        # STUDENTS (40)
        # -----------------------------
        student_names = [
            "Rahul", "Anita", "Kumar", "Priya", "Arjun", "Sneha", "Vikram", "Divya",
            "Karthik", "Meena", "Rohan", "Pooja", "Sanjay", "Nisha", "Ajay", "Lavanya",
            "Manoj", "Keerthi", "Surya", "Harini", "Dinesh", "Aarthi", "Gokul", "Ishwarya",
            "Naveen", "Reshma", "Prakash", "Deepika", "Vignesh", "Swetha", "Akash", "Monika",
            "Bharath", "Janani", "Saravanan", "Ritika", "Yogesh", "Anu", "Madhan", "Kavya"
        ]

        students = [(101 + i, student_names[i]) for i in range(40)]

        cursor.executemany("""
            INSERT OR IGNORE INTO students (id, name)
            VALUES (?, ?)
        """, students)

        # -----------------------------
        # SUBJECTS
        # -----------------------------
        subjects = [
            (1, 'Maths'),
            (2, 'Tamil'),
            (3, 'Science'),
            (4, 'Social'),
            (5, 'English'),
            (6, 'AI')
        ]

        cursor.executemany("""
            INSERT OR IGNORE INTO subjects (id, name)
            VALUES (?, ?)
        """, subjects)

        # -----------------------------
        # EXAM TYPES
        # -----------------------------
        exam_types = [
            (1, 'Unit Test', 20),
            (2, 'Midterm', 40),
            (3, 'Final Exam', 100)
        ]

        cursor.executemany("""
            INSERT OR IGNORE INTO exam_types (id, name, max_marks)
            VALUES (?, ?, ?)
        """, exam_types)

        # -----------------------------
        # EXAMS
        # -----------------------------
        exams = [
            (1, 'Unit Test 1', 1, '2024-01-10'),
            (2, 'Unit Test 2', 1, '2024-02-10'),
            (3, 'Unit Test 3', 1, '2024-03-10'),
            (4, 'Pre Midterm', 2, '2024-06-15'),
            (5, 'Post Midterm', 2, '2024-07-15'),
            (6, 'Half Yearly', 3, '2024-09-10'),
            (7, 'Annual Exam', 3, '2025-02-10')
        ]

        cursor.executemany("""
            INSERT OR IGNORE INTO exams (id, name, exam_type_id, date)
            VALUES (?, ?, ?, ?)
        """, exams)

        # -----------------------------
        # MARKS + MARK_COMPONENTS
        # -----------------------------
        marks_data = []
        mark_components_data = []

        for student_id, _ in students:
            for subject_id, _ in subjects:
                for exam_id, exam_name, exam_type_id, _ in exams:

                    # Score ranges by exam type
                    if exam_type_id == 1:  # Unit Test
                        total = random.randint(5, 20)

                    elif exam_type_id == 2:  # Midterm
                        total = random.randint(15, 40)

                    else:  # Final Exam
                        total = random.randint(35, 100)

                    # Insert mark
                    marks_data.append((student_id, subject_id, exam_id, total))

        cursor.executemany("""
            INSERT OR IGNORE INTO marks (student_id, subject_id, exam_id, marks_obtained)
            VALUES (?, ?, ?, ?)
        """, marks_data)

        # Fetch inserted marks IDs for final exams only
        cursor.execute("""
            SELECT id, marks_obtained
            FROM marks
            WHERE exam_id IN (6, 7)
        """)

        final_marks = cursor.fetchall()

        for mark_id, total in final_marks:

            # Written exam = 60–80
            written = min(random.randint(40, 80), total)

            remaining = total - written

            # Split remaining into 4 components
            class_test = min(random.randint(0, 5), remaining)
            remaining -= class_test

            activity = min(random.randint(0, 5), remaining)
            remaining -= activity

            assignment = min(random.randint(0, 5), remaining)
            remaining -= assignment

            project = remaining

            components = [
                (mark_id, 'written_exam', 80, written),
                (mark_id, 'class_test', 5, class_test),
                (mark_id, 'activity', 5, activity),
                (mark_id, 'assignment', 5, assignment),
                (mark_id, 'project', 5, project)
            ]

            mark_components_data.extend(components)

        cursor.executemany("""
            INSERT OR IGNORE INTO mark_components
            (mark_id, component_name, max_marks, marks_obtained)
            VALUES (?, ?, ?, ?)
        """, mark_components_data)

        # -----------------------------
        # ATTENDANCE (12 months)
        # -----------------------------
        attendance_data = []

        months = [f"2024-{str(m).zfill(2)}" for m in range(1, 13)]

        for student_id, _ in students:
            for month in months:
                working_days = random.randint(20, 24)
                present_days = random.randint(working_days - 6, working_days)
                percentage = round((present_days / working_days) * 100, 1)

                attendance_data.append(
                    (student_id, month, working_days, present_days, percentage)
                )

        cursor.executemany("""
            INSERT OR IGNORE INTO attendance
            (student_id, month, working_days, present_days, percentage)
            VALUES (?, ?, ?, ?, ?)
        """, attendance_data)

        conn.commit()

        print("Production-ready data inserted successfully.")
        print(f"Students: {len(students)}")
        print(f"Marks: {len(marks_data)}")
        print(f"Mark Components: {len(mark_components_data)}")
        print(f"Attendance Rows: {len(attendance_data)}")

    except Error as e:
        print(f"Error inserting mock data: {e}")
        conn.rollback()

    except Exception as e:
        print(f"Unexpected error: {e}")
        conn.rollback()

In [10]:
def simple_test_tables(conn):
    """Simple validation of all tables using basic queries"""
    try:
        cursor = conn.cursor()

        tables = ['students', 'subjects', 'exam_types', 'exams', 'marks']

        for table in tables:
            print(f"\n--- Testing {table} ---")

            # 1. Check if table has data
            cursor.execute(f"SELECT COUNT(*) FROM {table}")
            count = cursor.fetchone()[0]
            print(f"Total records: {count}")

            # 2. Fetch first 3 rows
            cursor.execute(f"SELECT * FROM {table} LIMIT 3")
            rows = cursor.fetchall()

            if rows:
                print("Sample data:")
                for row in rows:
                    print(row)
            else:
                print("No data found")

        print("\n✅ Simple testing completed successfully")

    except Exception as e:
        print(f"Error during simple testing: {e}")

In [11]:


def initialize_database(db_file="school.db"):
    """Main method to initialize DB"""
    try:
        conn = create_connection(db_file)
        print("Connection is created.")
        if conn is not None:
            print("Connection is not None.")
            create_tables(conn)
            insert_mock_data(conn)
            # If the DB already existed (previous runs), remove any accidental duplicates.
            conn.execute("""
            DELETE FROM marks
            WHERE id NOT IN (
                SELECT MIN(id)
                FROM marks
                GROUP BY student_id, subject_id, exam_id
            );
            """)
            conn.commit()
            simple_test_tables(conn)
        else:
            print("Failed to create database connection.")

    except Exception as e:
        print(f"Unexpected error: {e}")

    finally:
        if conn:
            conn.close()
            print("Database connection closed.")

In [12]:
initialize_database(db_file="school.db")

Database is created.
Connection is created.
Connection is not None.
Create table enter ...
Tables created successfully (if not already existing).
Production-ready data inserted successfully.
Students: 40
Marks: 1680
Mark Components: 2400
Attendance Rows: 480

--- Testing students ---
Total records: 40
Sample data:
(101, 'Rahul')
(102, 'Anita')
(103, 'Kumar')

--- Testing subjects ---
Total records: 6
Sample data:
(1, 'Maths')
(2, 'Tamil')
(3, 'Science')

--- Testing exam_types ---
Total records: 3
Sample data:
(1, 'Unit Test', 20)
(2, 'Midterm', 40)
(3, 'Final Exam', 100)

--- Testing exams ---
Total records: 7
Sample data:
(1, 'Unit Test 1', 1, '2024-01-10')
(2, 'Unit Test 2', 1, '2024-02-10')
(3, 'Unit Test 3', 1, '2024-03-10')

--- Testing marks ---
Total records: 1680
Sample data:
(1, 101, 1, 1, 18.0)
(2, 101, 1, 2, 17.0)
(3, 101, 1, 3, 19.0)

✅ Simple testing completed successfully
Database connection closed.


In [13]:
from collections import defaultdict

def get_connection(db_file: str = DB_FILE):
    # Keep connections short-lived for notebook/tool calls.
    conn = create_connection(db_file)
    if conn is None:
        raise RuntimeError(f"Failed to connect to database: {db_file}")
    return conn


def get_marks_history(student_id):
    logging.info(f"Fetching marks history for student_id={student_id}")

    conn = get_connection()
    try:
        cursor = conn.cursor()

        query = """ SELECT DISTINCT
    s.name AS subject,
    e.name AS exam,
    et.name AS exam_type,
    m.marks_obtained,
    et.max_marks,
    e.date
FROM marks m
JOIN subjects s ON m.subject_id = s.id
JOIN exams e ON m.exam_id = e.id
JOIN exam_types et ON e.exam_type_id = et.id
WHERE m.student_id = ?
ORDER BY e.date ASC """

        cursor.execute(query, (student_id,))
        rows = cursor.fetchall()

        if not rows:
            logging.warning("No marks found")
            return {"error": "No marks found"}

        data = defaultdict(list)

        for subject, exam, exam_type, score, max_marks, date in rows:
            data[subject].append({
                "exam": exam,
                "exam_type": exam_type,
                "score": score,
                "max_marks": max_marks,
                "date": date,
            })

        logging.info("Marks history fetched successfully")

        return {
            "student_id": student_id,
            "marks_history": dict(data),
        }
    finally:
        conn.close()

In [14]:
get_marks_history(101)

Database is created.


{'student_id': 101,
 'marks_history': {'Maths': [{'exam': 'Unit Test 1',
    'exam_type': 'Unit Test',
    'score': 18.0,
    'max_marks': 20,
    'date': '2024-01-10'},
   {'exam': 'Unit Test 2',
    'exam_type': 'Unit Test',
    'score': 17.0,
    'max_marks': 20,
    'date': '2024-02-10'},
   {'exam': 'Unit Test 3',
    'exam_type': 'Unit Test',
    'score': 19.0,
    'max_marks': 20,
    'date': '2024-03-10'},
   {'exam': 'Pre Midterm',
    'exam_type': 'Midterm',
    'score': 28.0,
    'max_marks': 40,
    'date': '2024-06-15'},
   {'exam': 'Post Midterm',
    'exam_type': 'Midterm',
    'score': 24.0,
    'max_marks': 40,
    'date': '2024-07-15'},
   {'exam': 'Half Yearly',
    'exam_type': 'Final Exam',
    'score': 78.0,
    'max_marks': 100,
    'date': '2024-09-10'},
   {'exam': 'Annual Exam',
    'exam_type': 'Final Exam',
    'score': 66.0,
    'max_marks': 100,
    'date': '2025-02-10'}],
  'Tamil': [{'exam': 'Unit Test 1',
    'exam_type': 'Unit Test',
    'score': 14.0,

In [15]:
def get_latest_marks(student_id):
    logging.info(f"Fetching latest marks for student_id={student_id}")

    conn = get_connection()
    try:
        cursor = conn.cursor()

        # Latest exam per subject for this student.
        query = """
WITH ranked AS (
    SELECT
        s.name AS subject,
        e.name AS exam,
        et.name AS exam_type,
        m.marks_obtained,
        et.max_marks,
        e.date,
        ROW_NUMBER() OVER (
            PARTITION BY s.id
            ORDER BY e.date DESC, e.id DESC
        ) AS rn
    FROM marks m
    JOIN subjects s ON m.subject_id = s.id
    JOIN exams e ON m.exam_id = e.id
    JOIN exam_types et ON e.exam_type_id = et.id
    WHERE m.student_id = ?
)
SELECT subject, exam, exam_type, marks_obtained, max_marks, date
FROM ranked
WHERE rn = 1
ORDER BY subject ASC;
"""

        cursor.execute(query, (student_id,))
        rows = cursor.fetchall()
    finally:
        conn.close()

    if not rows:
        logging.warning("No latest marks found")
        return {"error": "No marks found"}

    data = {}

    for subject, exam, exam_type, score, max_marks, date in rows:
        data[subject] = {
            "exam": exam,
            "exam_type": exam_type,
            "score": score,
            "max_marks": max_marks,
            "date": date,
        }

    logging.info("Latest marks fetched successfully")

    return {
        "student_id": student_id,
        "latest_marks": data,
    }
    if not rows:
        logging.warning("No latest marks found")
        return {"error": "No marks found"}

    data = {}

    for subject, exam, exam_type, score, max_marks, date in rows:
        data[subject] = {
            "exam": exam,
            "exam_type": exam_type,
            "score": score,
            "max_marks": max_marks,
            "date": date
        }

    logging.info("Latest marks fetched successfully")

    return {
        "student_id": student_id,
        "latest_marks": data
    }

In [16]:
get_latest_marks(101)

Database is created.


{'student_id': 101,
 'latest_marks': {'AI': {'exam': 'Annual Exam',
   'exam_type': 'Final Exam',
   'score': 79.0,
   'max_marks': 100,
   'date': '2025-02-10'},
  'English': {'exam': 'Annual Exam',
   'exam_type': 'Final Exam',
   'score': 48.0,
   'max_marks': 100,
   'date': '2025-02-10'},
  'Maths': {'exam': 'Annual Exam',
   'exam_type': 'Final Exam',
   'score': 66.0,
   'max_marks': 100,
   'date': '2025-02-10'},
  'Science': {'exam': 'Annual Exam',
   'exam_type': 'Final Exam',
   'score': 43.0,
   'max_marks': 100,
   'date': '2025-02-10'},
  'Social': {'exam': 'Annual Exam',
   'exam_type': 'Final Exam',
   'score': 37.0,
   'max_marks': 100,
   'date': '2025-02-10'},
  'Tamil': {'exam': 'Annual Exam',
   'exam_type': 'Final Exam',
   'score': 72.0,
   'max_marks': 100,
   'date': '2025-02-10'}}}

In [17]:
def get_marks(student_id):
    logging.info(f"Fallback get_marks called for student_id={student_id}")
    return get_marks_history(student_id)

In [18]:
get_marks(135)

Database is created.


{'student_id': 135,
 'marks_history': {'Maths': [{'exam': 'Unit Test 1',
    'exam_type': 'Unit Test',
    'score': 15.0,
    'max_marks': 20,
    'date': '2024-01-10'},
   {'exam': 'Unit Test 2',
    'exam_type': 'Unit Test',
    'score': 16.0,
    'max_marks': 20,
    'date': '2024-02-10'},
   {'exam': 'Unit Test 3',
    'exam_type': 'Unit Test',
    'score': 7.0,
    'max_marks': 20,
    'date': '2024-03-10'},
   {'exam': 'Pre Midterm',
    'exam_type': 'Midterm',
    'score': 34.0,
    'max_marks': 40,
    'date': '2024-06-15'},
   {'exam': 'Post Midterm',
    'exam_type': 'Midterm',
    'score': 35.0,
    'max_marks': 40,
    'date': '2024-07-15'},
   {'exam': 'Half Yearly',
    'exam_type': 'Final Exam',
    'score': 75.0,
    'max_marks': 100,
    'date': '2024-09-10'},
   {'exam': 'Annual Exam',
    'exam_type': 'Final Exam',
    'score': 75.0,
    'max_marks': 100,
    'date': '2025-02-10'}],
  'Tamil': [{'exam': 'Unit Test 1',
    'exam_type': 'Unit Test',
    'score': 9.0,
 

In [19]:
import logging
from collections import defaultdict

def get_mark_components(student_id):
    logging.info(f"Fetching mark components for student_id={student_id}")

    conn = get_connection()
    cursor = conn.cursor()

    query = """
    SELECT 
        s.name AS subject,
        e.name AS exam,
        mc.component_name,
        mc.max_marks,
        mc.marks_obtained
    FROM mark_components mc
    JOIN marks m ON mc.mark_id = m.id
    JOIN subjects s ON m.subject_id = s.id
    JOIN exams e ON m.exam_id = e.id
    WHERE m.student_id = ?
    ORDER BY e.date ASC
    """

    cursor.execute(query, (student_id,))
    rows = cursor.fetchall()
    conn.close()

    if not rows:
        return {"error": "No component data found"}

    data = defaultdict(lambda: defaultdict(list))

    for subject, exam, component, max_marks, obtained in rows:
        data[subject][exam].append({
            "component": component,
            "max_marks": max_marks,
            "marks_obtained": obtained
        })

    return {
        "student_id": student_id,
        "mark_components": {
            subject: dict(exams)
            for subject, exams in data.items()
        }
    }

In [20]:
get_mark_components(101)

Database is created.


{'student_id': 101,
 'mark_components': {'Maths': {'Half Yearly': [{'component': 'written_exam',
     'max_marks': 80.0,
     'marks_obtained': 62.0},
    {'component': 'class_test', 'max_marks': 5.0, 'marks_obtained': 5.0},
    {'component': 'activity', 'max_marks': 5.0, 'marks_obtained': 4.0},
    {'component': 'assignment', 'max_marks': 5.0, 'marks_obtained': 4.0},
    {'component': 'project', 'max_marks': 5.0, 'marks_obtained': 3.0},
    {'component': 'written_exam', 'max_marks': 80.0, 'marks_obtained': 62.0},
    {'component': 'class_test', 'max_marks': 5.0, 'marks_obtained': 5.0},
    {'component': 'activity', 'max_marks': 5.0, 'marks_obtained': 4.0},
    {'component': 'assignment', 'max_marks': 5.0, 'marks_obtained': 4.0},
    {'component': 'project', 'max_marks': 5.0, 'marks_obtained': 3.0},
    {'component': 'written_exam', 'max_marks': 80.0, 'marks_obtained': 62.0},
    {'component': 'class_test', 'max_marks': 5.0, 'marks_obtained': 5.0},
    {'component': 'activity', 'max_ma

In [21]:
def get_attendance(student_id):
    logging.info(f"Fetching attendance for student_id={student_id}")

    conn = get_connection()
    cursor = conn.cursor()

    query = """
    SELECT month, working_days, present_days, percentage
    FROM attendance
    WHERE student_id = ?
    ORDER BY month ASC
    """

    cursor.execute(query, (student_id,))
    rows = cursor.fetchall()
    conn.close()

    if not rows:
        return {"error": "No attendance data found"}

    attendance_data = []

    for month, working_days, present_days, percentage in rows:
        attendance_data.append({
            "month": month,
            "working_days": working_days,
            "present_days": present_days,
            "percentage": percentage
        })

    return {
        "student_id": student_id,
        "attendance": attendance_data
    }

In [22]:
get_attendance(101)

Database is created.


{'student_id': 101,
 'attendance': [{'month': '2024-01',
   'working_days': 22,
   'present_days': 18,
   'percentage': 81.8},
  {'month': '2024-01',
   'working_days': 22,
   'present_days': 18,
   'percentage': 81.8},
  {'month': '2024-01',
   'working_days': 22,
   'present_days': 18,
   'percentage': 81.8},
  {'month': '2024-01',
   'working_days': 22,
   'present_days': 18,
   'percentage': 81.8},
  {'month': '2024-01',
   'working_days': 22,
   'present_days': 18,
   'percentage': 81.8},
  {'month': '2024-01',
   'working_days': 22,
   'present_days': 18,
   'percentage': 81.8},
  {'month': '2024-01',
   'working_days': 23,
   'present_days': 18,
   'percentage': 78.3},
  {'month': '2024-01',
   'working_days': 22,
   'present_days': 22,
   'percentage': 100.0},
  {'month': '2024-02',
   'working_days': 20,
   'present_days': 15,
   'percentage': 75.0},
  {'month': '2024-02',
   'working_days': 20,
   'present_days': 15,
   'percentage': 75.0},
  {'month': '2024-02',
   'working_

In [17]:
import logging
import re

def normalize_name(name):
    """
    Normalize names for flexible comparison:
    - lowercase
    - remove extra spaces
    - remove punctuation
    """
    name = name.lower().strip()
    name = re.sub(r"[^\w\s]", "", name)   # remove punctuation
    name = re.sub(r"\s+", " ", name)      # collapse spaces
    return name


def resolve_student(name_or_id):
    logging.info(f"Resolving student input: {name_or_id}")

    conn = get_connection()
    cursor = conn.cursor()

    try:
        if name_or_id is None:
            return {"error": "Student input is required"}

        raw_input = str(name_or_id).strip()

        if not raw_input:
            return {"error": "Student input cannot be empty"}

        # CASE 1: Numeric ID
        if raw_input.isdigit():
            logging.info("Detected numeric ID")

            cursor.execute("""
                SELECT id, name
                FROM students
                WHERE id = ?
            """, (int(raw_input),))

            row = cursor.fetchone()

            if not row:
                return {"error": f"Student ID '{raw_input}' not found"}

            return {
                "student_id": row[0],
                "name": row[1]
            }

        # CASE 2: Name search
        normalized_input = normalize_name(raw_input)

        cursor.execute("SELECT id, name FROM students")
        students = cursor.fetchall()

        exact_matches = []
        startswith_matches = []
        partial_matches = []

        input_tokens = normalized_input.split()

        for student_id, student_name in students:
            normalized_db_name = normalize_name(student_name)
            db_tokens = normalized_db_name.split()

            # Exact full match
            if normalized_db_name == normalized_input:
                exact_matches.append((student_id, student_name))

            # Startswith full name
            elif normalized_db_name.startswith(normalized_input):
                startswith_matches.append((student_id, student_name))

            # Token-based partial match
            elif any(token in db_tokens or token in normalized_db_name for token in input_tokens):
                partial_matches.append((student_id, student_name))

        # PRIORITY 1: Exact
        if len(exact_matches) == 1:
            return {
                "student_id": exact_matches[0][0],
                "name": exact_matches[0][1]
            }

        # PRIORITY 2: Startswith
        if len(exact_matches) == 0 and len(startswith_matches) == 1:
            return {
                "student_id": startswith_matches[0][0],
                "name": startswith_matches[0][1]
            }

        # PRIORITY 3: Partial single
        if len(exact_matches) == 0 and len(startswith_matches) == 0 and len(partial_matches) == 1:
            return {
                "student_id": partial_matches[0][0],
                "name": partial_matches[0][1]
            }

        # MULTIPLE MATCHES
        all_matches = exact_matches or startswith_matches or partial_matches

        if len(all_matches) > 1:
            return {
                "multiple_matches": [
                    {
                        "student_id": sid,
                        "name": sname
                    }
                    for sid, sname in all_matches[:10]
                ],
                "message": "Multiple students found. Please refine your search."
            }

        # NO MATCH
        return {
            "error": f"No student found matching '{raw_input}'"
        }

    except Exception as e:
        logging.error(f"Resolve student failed: {str(e)}")
        return {"error": str(e)}

    finally:
        conn.close()

In [20]:
import sqlite3

conn = sqlite3.connect("school.db")
cursor = conn.cursor()

cursor.execute("SELECT * FROM students where name = 'Rahul'")

# Get column headers
headers = [column[0] for column in cursor.description]

# Print headers
print(" | ".join(headers))
print("-" * 30)

# Print rows
for row in cursor.fetchall():
    print(" | ".join(str(value) for value in row))

conn.close()

id | name
------------------------------
101 | Rahul


In [24]:
# IMPORTANT:
# - `TOOLS_SCHEMA` (below) is sent to the model -> must be JSON-serializable.
# - Actual Python callables live in `TOOLS_MAP` and are executed locally.

TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "get_marks_history",
            "description": "Use when user asks for trends, past performance, or improvement over time",
            "parameters": {
                "type": "object",
                "properties": {"student_id": {"type": "string"}},
                "required": ["student_id"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_latest_marks",
            "description": "Use when user asks for latest performance, current marks, or most recent exam",
            "parameters": {
                "type": "object",
                "properties": {"student_id": {"type": "string"}},
                "required": ["student_id"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_marks",
            "description": "Fallback tool for general marks queries",
            "parameters": {
                "type": "object",
                "properties": {"student_id": {"type": "string"}},
                "required": ["student_id"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
        "name": "get_mark_components",
        "description": "Use when user asks for exam breakdown, internal marks, activity marks, class test contribution, or score composition",
        "parameters": {
            "type": "object",
            "properties": {
            "student_id": {
                "type": "string",
                "description": "Unique identifier of the student"
            }
            },
            "required": ["student_id"],
            "additionalProperties": False
        }
        }
    },
    {
        "type": "function",
        "function": {
        "name": "get_attendance",
        "description": "Use when user asks about attendance, consistency, absenteeism, or school presence",
        "parameters": {
            "type": "object",
            "properties": {
            "student_id": {
                "type": "string",
                "description": "Unique identifier of the student"
            }
            },
            "required": ["student_id"],
            "additionalProperties": False
        }
        }
    }
]

print(TOOLS_SCHEMA)

[{'type': 'function', 'function': {'name': 'get_marks_history', 'description': 'Use when user asks for trends, past performance, or improvement over time', 'parameters': {'type': 'object', 'properties': {'student_id': {'type': 'string'}}, 'required': ['student_id'], 'additionalProperties': False}}}, {'type': 'function', 'function': {'name': 'get_latest_marks', 'description': 'Use when user asks for latest performance, current marks, or most recent exam', 'parameters': {'type': 'object', 'properties': {'student_id': {'type': 'string'}}, 'required': ['student_id'], 'additionalProperties': False}}}, {'type': 'function', 'function': {'name': 'get_marks', 'description': 'Fallback tool for general marks queries', 'parameters': {'type': 'object', 'properties': {'student_id': {'type': 'string'}}, 'required': ['student_id'], 'additionalProperties': False}}}, {'type': 'function', 'function': {'name': 'get_mark_components', 'description': 'Use when user asks for exam breakdown, internal marks, ac

In [25]:
TOOLS_MAP = {
    "get_latest_marks": get_latest_marks,
    "get_marks_history": get_marks_history,
    "get_marks": get_marks,
    "get_mark_components": get_mark_components,
    "get_attendance": get_attendance,
}

In [26]:
import json

def handle_tool_calls(assistant_msg):
    """Convert an assistant tool request into tool result messages."""

    tool_calls = getattr(assistant_msg, "tool_calls", None) or []
    tool_messages = []
    print("chat 3",tool_calls)
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        if tool_name not in TOOLS_MAP:
            raise ValueError(f"Unknown tool: {tool_name}")
       
        arguments = json.loads(tool_call.function.arguments or "{}")
        result = TOOLS_MAP[tool_name](**arguments)
        print("chat 4",result)
        tool_messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result, default=str),
            }
        )

    return tool_messages

In [27]:
import json

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = openrouter.chat.completions.create(
        model=gemma4,
        messages=messages,
        tools=TOOLS_SCHEMA,
        tool_choice="auto"
    )
    print("call 1",response.choices[0].message)
    while True:
        assistant_msg = response.choices[0].message
        tool_calls = getattr(assistant_msg, "tool_calls", None)

        if tool_calls:
            # 1) add assistant tool-request message
            messages.append(
                {
                    "role": "assistant",
                    "content": assistant_msg.content,
                    "tool_calls": [
                        {
                            "id": tc.id,
                            "type": "function",
                            "function": {
                                "name": tc.function.name,
                                "arguments": tc.function.arguments,
                            },
                        }
                        for tc in tool_calls
                    ],
                }
            )
            print("Call 2",messages)
            # 2) execute tools via helper
            messages.extend(handle_tool_calls(assistant_msg))

            # 3) call model again with tool outputs
            response = openrouter.chat.completions.create(
                model=gemma4,
                messages=messages,
                tools=TOOLS_SCHEMA,
                tool_choice="auto"
            )
        else:
            return assistant_msg.content or ""

In [28]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


call 1 ChatCompletionMessage(content="Hello! I am here to help you analyze your students' performance for your parent-teacher meetings. Please provide the student's ID and let me know what specific information you need (e.g., latest marks, performance trends, or attendance).", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning=None)
call 1 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-9e7d5eef82e6b9f1', function=Function(arguments='{"student_id": "101"}', name='get_latest_marks'), type='function', index=0)], reasoning=None)
Call 2 [{'role': 'system', 'content': 'You are an AI assistant designed to help teachers analyze student performance for parent-teacher meetings.\n\nYour responsibilities:\n- Understand the teacher’s question accurately\n- Determine what type of academic data is needed\n-